# Objective benchmark

One locked cross-user split; model seeds 11/23/37 measure optimization variability, not uncertainty across user splits. Shuffle replicates are averaged within each model seed before mean/SD. The REF checkpoint is reused, not retrained or counted as a new replicate. Primary representation is communicated spikes; pre-reset membrane is secondary. Relative10 is an offline, duration-aware reference, not a guaranteed upper bound. No training or probe fitting is performed by this notebook.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
import pandas as pd

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core_benchmark_v1/protocol.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Open this notebook inside the WritingRing repository")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from core_benchmark_v1.analysis import results_root, load_manifest, table, plot_native, plot_probe, plot_readout
ROOT = results_root(REPO)
MANIFEST = load_manifest(ROOT)
display(MANIFEST)
display(ROOT)


## Native accumulator

O0: L2 WCCE. O1: L2 TSCE. O2/O3: L2 WCCE plus 0.1 L1 WCCE/TSCE. Auxiliary heads are training-only. WCCE uses valid-mean logits; TSCE averages within each sample, then over samples.

In [ ]:
native = table(ROOT, "native_summary", "01_objective")
display(native)
plot_native(native, "Objective benchmark")

## Paired contrasts

Deltas are paired by model seed. Inspect every seed, not only the average.

In [ ]:
paired = table(ROOT, "paired_contrasts")
display(paired[paired.block == "01_objective"] if not paired.empty else paired)

## Layer-wise probes

Identical scale-only preprocessing for affine/no-bias probes; C selected using validation users only.

In [ ]:
probes = table(ROOT, "probe_summary", "01_objective")
display(probes)
plot_probe(probes, "Primary spike/no-bias probes")

## Temporal and geometry gains

G_resolved = Fixed250 ordered - WholeCount; G_order = ordered - shuffled; G_geometry = affine - no-bias. These measure decoder-accessible organization, not pure mutual information.

In [ ]:
display(table(ROOT, "probe_gains_summary", "01_objective"))

## Activity and generalization gap

Interpret tau/depth changes together with firing rate, silent neurons, membrane scale and the native/probe train-test gaps.

In [ ]:
display(table(ROOT, "activity"))